# NOODLE: Uncertainty-Aware Hardware Trojan Detection Using Multimodal Deep Learning

## LLM4ChipDesign - Undergraduate Course Lab

![NOODLE Framework](https://github.com/cars-lab-repo/NOODLE/assets/64368687/8e56628b-ec27-4e8a-a87f-dfc164e25dbd)

### Course Objectives:
- Understand multimodal deep learning for hardware security
- Learn to detect hardware Trojans using graph and tabular data
- Implement early and late fusion strategies
- Evaluate uncertainty quantification in security predictions
- Apply machine learning techniques to real-world chip design security

### Authors: 
- [Rahul Vishwakarma](https://github.com/rahvis) 
- [Amin Rezaei](https://github.com/r3zaei)

---

**Important**: This notebook is designed to run on Google Colab. Make sure to upload the dataset files or clone the repository when prompted.

## 🚀 Section 1: Environment Setup and Data Preparation

In this section, we'll set up our Google Colab environment and prepare the NOODLE dataset for hardware Trojan detection.

In [ ]:
# Install required packages for the NOODLE framework
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install scikit-learn pandas numpy matplotlib seaborn
!pip install torch-geometric
!pip install networkx
!pip install plotly

print("✅ All required packages installed successfully!")

In [ ]:
# Clone the NOODLE repository or upload files
import os
import subprocess

try:
    # Try to clone the repository
    if not os.path.exists('NOODLE'):
        !git clone https://github.com/cars-lab-repo/NOODLE.git
        print("✅ Repository cloned successfully!")
    else:
        print("✅ Repository already exists!")
except:
    print("⚠️  If cloning failed, please manually upload the dataset files:")
    print("   - Upload 'synthetic-multimodal-graph_and_table' folder")
    print("   - Upload 'source' folder with Python scripts")
    print("   - Or create sample data (we'll provide fallback code)")

# Change to NOODLE directory if it exists
if os.path.exists('NOODLE'):
    os.chdir('NOODLE')
    
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, brier_score_loss, 
    confusion_matrix, classification_report, accuracy_score
)
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 📊 Section 2: Data Loading and Exploration

Let's explore both modalities of our dataset:
1. **Tabular Data**: Statistical features extracted from hardware designs
2. **Graph Data**: Structural representation of circuit connections

In [ ]:
# Create synthetic dataset if original files are not available
def create_synthetic_data():
    """Create synthetic multimodal dataset for demonstration"""
    print("Creating synthetic dataset for demonstration...")
    
    # Create synthetic tabular data
    np.random.seed(42)
    n_samples = 1000
    n_features = 16
    
    # Generate features for Trojan-free samples (50%)
    trojan_free_features = np.random.normal(25, 10, (n_samples//2, n_features))
    trojan_free_labels = np.zeros(n_samples//2)
    
    # Generate features for Trojan-infected samples (50%)
    trojan_infected_features = np.random.normal(35, 12, (n_samples//2, n_features))
    trojan_infected_labels = np.ones(n_samples//2)
    
    # Combine data
    X_tabular = np.vstack([trojan_free_features, trojan_infected_features])
    y = np.hstack([trojan_free_labels, trojan_infected_labels])
    
    # Create DataFrame
    columns = [f'Feature{i+1}' for i in range(n_features)] + ['Trojan']
    tabular_data = pd.DataFrame(
        np.column_stack([X_tabular, y]), 
        columns=columns
    )
    
    # Create synthetic graph data (simplified)
    graph_features = X_tabular + np.random.normal(0, 5, X_tabular.shape)
    graph_data = pd.DataFrame(
        np.column_stack([graph_features, y]), 
        columns=columns
    )
    
    return tabular_data, graph_data

# Try to load real data, fallback to synthetic
try:
    # Load tabular data
    tabular_path = 'synthetic-multimodal-graph_and_table/dataset/tabular/tabular_dataset.csv'
    if os.path.exists(tabular_path):
        tabular_data = pd.read_csv(tabular_path)
        print("✅ Loaded real tabular dataset")
    else:
        raise FileNotFoundError("Real dataset not found")
        
except:
    print("⚠️  Real dataset not found. Creating synthetic data...")
    tabular_data, graph_data = create_synthetic_data()

print(f"Dataset shape: {tabular_data.shape}")
print(f"Features: {list(tabular_data.columns[:-1])}")
print(f"Target variable: {tabular_data.columns[-1]}")

# Display basic statistics
print("\n📈 Dataset Overview:")
print(tabular_data.head())

In [ ]:
# Explore the data distribution
plt.figure(figsize=(15, 5))

# Plot 1: Label distribution
plt.subplot(1, 3, 1)
label_counts = tabular_data.iloc[:, -1].value_counts()
plt.pie(label_counts.values, labels=['Trojan-Free', 'Trojan-Infected'], autopct='%1.1f%%', startangle=90)
plt.title('Hardware Trojan Distribution')

# Plot 2: Feature correlation heatmap
plt.subplot(1, 3, 2)
correlation_matrix = tabular_data.iloc[:, :-1].corr()
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Matrix')

# Plot 3: Feature distributions by class
plt.subplot(1, 3, 3)
feature_means_by_class = tabular_data.groupby(tabular_data.columns[-1]).mean()
feature_means_by_class.T.plot(kind='bar', ax=plt.gca())
plt.title('Average Features by Trojan Status')
plt.xticks(rotation=45)
plt.legend(['Trojan-Free', 'Trojan-Infected'])

plt.tight_layout()
plt.show()

# Print class distribution
print("\n📊 Class Distribution:")
print(f"Trojan-Free: {(tabular_data.iloc[:, -1] == 0).sum()} samples")
print(f"Trojan-Infected: {(tabular_data.iloc[:, -1] == 1).sum()} samples")

## 🧠 Section 3: Tabular Data Analysis with CNN

Let's implement our first modality - analyzing tabular hardware features using a 1D Convolutional Neural Network.

In [ ]:
# Define the Tabular CNN Model
class TabularCNN(nn.Module):
    def __init__(self, input_features):
        super(TabularCNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.dropout = nn.Dropout(0.3)
        
        # Calculate the size after convolution and pooling
        conv_output_size = 64 * (input_features // 2)
        self.fc1 = nn.Linear(conv_output_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = self.dropout(x)
        
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

print("✅ Tabular CNN model defined!")

In [ ]:
# Prepare tabular data for training
def prepare_tabular_data(data):
    """Prepare tabular data for CNN model"""
    X = data.iloc[:, :-1].values.astype(np.float32)
    y = data.iloc[:, -1].values.astype(np.float32)
    
    # Normalize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Reshape for 1D CNN (batch_size, channels, features)
    X_reshaped = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_reshaped, y, test_size=0.2, random_state=42, stratify=y
    )
    
    return X_train, X_test, y_train, y_test, scaler

# Prepare the data
X_train, X_test, y_train, y_test, scaler = prepare_tabular_data(tabular_data)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Feature dimension: {X_train.shape[2]}")

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train)
y_test_tensor = torch.FloatTensor(y_test)

print("✅ Data prepared for training!")

In [ ]:
# Train the Tabular CNN Model
def train_model(model, X_train, y_train, X_test, y_test, epochs=50):
    """Train the model and return training history"""
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    
    # Create data loaders
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    
    model.train()
    for epoch in range(epochs):
        epoch_train_loss = 0
        epoch_train_correct = 0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            
            outputs = model(batch_X).squeeze()
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
            
            epoch_train_loss += loss.item()
            predictions = (outputs > 0.5).float()
            epoch_train_correct += (predictions == batch_y).sum().item()
        
        # Calculate metrics
        avg_train_loss = epoch_train_loss / len(train_loader)
        train_acc = epoch_train_correct / len(X_train)
        
        # Evaluate on test set
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test).squeeze()
            test_loss = criterion(test_outputs, y_test).item()
            test_predictions = (test_outputs > 0.5).float()
            test_acc = (test_predictions == y_test).float().mean().item()
        
        train_losses.append(avg_train_loss)
        test_losses.append(test_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        
        if epoch % 10 == 0:
            print(f'Epoch {epoch}/{epochs}: Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')
        
        model.train()
    
    return train_losses, test_losses, train_accs, test_accs

# Initialize and train the model
input_features = X_train.shape[2]
tabular_model = TabularCNN(input_features)

print("🚀 Training Tabular CNN Model...")
train_losses, test_losses, train_accs, test_accs = train_model(
    tabular_model, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor
)

In [ ]:
# Evaluate Tabular Model Performance
tabular_model.eval()
with torch.no_grad():
    # Get predictions
    train_predictions = tabular_model(X_train_tensor).squeeze().numpy()
    test_predictions = tabular_model(X_test_tensor).squeeze().numpy()
    
    # Calculate metrics
    train_auc = roc_auc_score(y_train, train_predictions)
    test_auc = roc_auc_score(y_test, test_predictions)
    
    test_pred_binary = (test_predictions > 0.5).astype(int)
    test_accuracy = accuracy_score(y_test, test_pred_binary)
    
    # Brier Score (uncertainty quantification)
    brier_score = brier_score_loss(y_test, test_predictions)

print("📊 Tabular Model Performance:")
print(f"Training AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Brier Score (lower is better): {brier_score:.4f}")

# Plot training curves and ROC curve
plt.figure(figsize=(15, 5))

# Training curves
plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.title('Training Curves - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(test_accs, label='Test Accuracy')
plt.title('Training Curves - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# ROC Curve
plt.subplot(1, 3, 3)
fpr, tpr, _ = roc_curve(y_test, test_predictions)
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {test_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Tabular Model')
plt.legend()

plt.tight_layout()
plt.show()

## 🔗 Section 4: Graph Data Analysis

Now let's implement the second modality - analyzing graph-structured hardware data representing circuit topology.

In [ ]:
# Create graph data (using the same approach as the original but simplified)
def create_graph_features(data):
    """
    Convert tabular data to graph-like features
    In practice, this would involve actual graph neural networks
    """
    # For this demo, we'll simulate graph features by applying different transformations
    np.random.seed(42)
    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].values
    
    # Simulate graph-based features (in practice, these would come from GNN embeddings)
    graph_features = X + np.random.normal(0, 2, X.shape)  # Add graph-specific noise
    
    # Apply some graph-inspired transformations
    # Simulate node degree, clustering coefficient, etc.
    for i in range(graph_features.shape[1]):
        if i % 2 == 0:  # Even indices: simulate degree-like features
            graph_features[:, i] = np.abs(graph_features[:, i])
        else:  # Odd indices: simulate connectivity features  
            graph_features[:, i] = np.tanh(graph_features[:, i])
    
    return graph_features, y

# Generate graph features
try:
    # Try to use existing graph data structure if available
    graph_features, graph_labels = create_graph_features(tabular_data)
    print("✅ Graph features created successfully!")
except:
    # Fallback to tabular data
    graph_features, graph_labels = create_graph_features(tabular_data)
    print("⚠️  Using simulated graph features")

print(f"Graph features shape: {graph_features.shape}")
print(f"Graph labels shape: {graph_labels.shape}")

# Visualize graph feature distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(graph_features[graph_labels==0].flatten(), alpha=0.5, label='Trojan-Free', bins=30)
plt.hist(graph_features[graph_labels==1].flatten(), alpha=0.5, label='Trojan-Infected', bins=30)
plt.title('Graph Feature Distribution by Class')
plt.xlabel('Feature Value')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 2, 2)
# Show average feature values
feature_means = [graph_features[graph_labels==0].mean(axis=0), 
                 graph_features[graph_labels==1].mean(axis=0)]
x_pos = np.arange(len(feature_means[0]))
plt.bar(x_pos - 0.2, feature_means[0], width=0.4, label='Trojan-Free', alpha=0.7)
plt.bar(x_pos + 0.2, feature_means[1], width=0.4, label='Trojan-Infected', alpha=0.7)
plt.title('Average Graph Features by Class')
plt.xlabel('Feature Index')
plt.ylabel('Average Value')
plt.legend()
plt.xticks(x_pos[::2])

plt.tight_layout()
plt.show()

In [ ]:
# Define Graph CNN Model (simplified version of GNN)
class GraphCNN(nn.Module):
    def __init__(self, input_features):
        super(GraphCNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.dropout = nn.Dropout(0.4)
        
        # Calculate the size after convolution and pooling
        conv_output_size = 128 * (input_features // 2)
        self.fc1 = nn.Linear(conv_output_size, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 1)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = self.dropout(x)
        
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

# Prepare graph data
def prepare_graph_data(features, labels):
    """Prepare graph data for training"""
    # Normalize features
    scaler_graph = StandardScaler()
    X_scaled = scaler_graph.fit_transform(features)
    
    # Reshape for 1D CNN
    X_reshaped = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])
    
    # Split data
    X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
        X_reshaped, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    return X_train_g, X_test_g, y_train_g, y_test_g, scaler_graph

# Prepare graph data
X_train_graph, X_test_graph, y_train_graph, y_test_graph, scaler_graph = prepare_graph_data(
    graph_features, graph_labels
)

print(f"Graph training set: {X_train_graph.shape}")
print(f"Graph test set: {X_test_graph.shape}")

# Convert to PyTorch tensors
X_train_graph_tensor = torch.FloatTensor(X_train_graph)
X_test_graph_tensor = torch.FloatTensor(X_test_graph)
y_train_graph_tensor = torch.FloatTensor(y_train_graph)
y_test_graph_tensor = torch.FloatTensor(y_test_graph)

# Initialize and train graph model
graph_model = GraphCNN(X_train_graph.shape[2])

print("🚀 Training Graph CNN Model...")
train_losses_g, test_losses_g, train_accs_g, test_accs_g = train_model(
    graph_model, X_train_graph_tensor, y_train_graph_tensor, 
    X_test_graph_tensor, y_test_graph_tensor
)

In [ ]:
# Evaluate Graph Model Performance
graph_model.eval()
with torch.no_grad():
    # Get predictions
    graph_train_pred = graph_model(X_train_graph_tensor).squeeze().numpy()
    graph_test_pred = graph_model(X_test_graph_tensor).squeeze().numpy()
    
    # Calculate metrics
    graph_train_auc = roc_auc_score(y_train_graph, graph_train_pred)
    graph_test_auc = roc_auc_score(y_test_graph, graph_test_pred)
    
    graph_test_pred_binary = (graph_test_pred > 0.5).astype(int)
    graph_test_accuracy = accuracy_score(y_test_graph, graph_test_pred_binary)
    
    # Brier Score
    graph_brier_score = brier_score_loss(y_test_graph, graph_test_pred)

print("📊 Graph Model Performance:")
print(f"Training AUC: {graph_train_auc:.4f}")
print(f"Test AUC: {graph_test_auc:.4f}")
print(f"Test Accuracy: {graph_test_accuracy:.4f}")
print(f"Brier Score: {graph_brier_score:.4f}")

# Plot graph model results
plt.figure(figsize=(15, 5))

# Training curves
plt.subplot(1, 3, 1)
plt.plot(train_losses_g, label='Train Loss')
plt.plot(test_losses_g, label='Test Loss')
plt.title('Graph Model - Training Curves (Loss)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(train_accs_g, label='Train Accuracy')
plt.plot(test_accs_g, label='Test Accuracy')
plt.title('Graph Model - Training Curves (Accuracy)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# ROC Curve
plt.subplot(1, 3, 3)
fpr_g, tpr_g, _ = roc_curve(y_test_graph, graph_test_pred)
plt.plot(fpr_g, tpr_g, label=f'Graph ROC (AUC = {graph_test_auc:.3f})', color='green')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Graph Model')
plt.legend()

plt.tight_layout()
plt.show()

## 🔄 Section 5: Early Fusion - Multimodal Learning

Early fusion combines features from both modalities before making predictions. This approach learns joint representations from the beginning.

In [ ]:
# Early Fusion Model
class EarlyFusionModel(nn.Module):
    def __init__(self, tabular_features, graph_features):
        super(EarlyFusionModel, self).__init__()
        
        # Separate feature extractors for each modality
        self.tabular_conv = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
        )
        
        self.graph_conv = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
        )
        
        # Calculate output sizes after convolution
        tabular_output_size = 64 * (tabular_features // 4)
        graph_output_size = 64 * (graph_features // 4)
        
        # Fusion layer
        fusion_input_size = tabular_output_size + graph_output_size
        self.fusion_layers = nn.Sequential(
            nn.Linear(fusion_input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, tabular_x, graph_x):
        # Process each modality
        tab_features = self.tabular_conv(tabular_x)
        graph_features = self.graph_conv(graph_x)
        
        # Flatten
        tab_features = tab_features.view(tab_features.size(0), -1)
        graph_features = graph_features.view(graph_features.size(0), -1)
        
        # Concatenate (Early Fusion)
        fused_features = torch.cat([tab_features, graph_features], dim=1)
        
        # Final prediction
        output = self.fusion_layers(fused_features)
        return output

print("✅ Early Fusion model defined!")

In [ ]:
# Train Early Fusion Model
def train_fusion_model(model, X_tab_train, X_graph_train, y_train, 
                      X_tab_test, X_graph_test, y_test, epochs=50):
    """Train the early fusion model"""
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    
    # Create combined dataset
    train_dataset = TensorDataset(X_tab_train, X_graph_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    
    model.train()
    for epoch in range(epochs):
        epoch_train_loss = 0
        epoch_train_correct = 0
        
        for batch_tab, batch_graph, batch_y in train_loader:
            optimizer.zero_grad()
            
            outputs = model(batch_tab, batch_graph).squeeze()
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
            
            epoch_train_loss += loss.item()
            predictions = (outputs > 0.5).float()
            epoch_train_correct += (predictions == batch_y).sum().item()
        
        # Calculate metrics
        avg_train_loss = epoch_train_loss / len(train_loader)
        train_acc = epoch_train_correct / len(X_tab_train)
        
        # Evaluate on test set
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_tab_test, X_graph_test).squeeze()
            test_loss = criterion(test_outputs, y_test).item()
            test_predictions = (test_outputs > 0.5).float()
            test_acc = (test_predictions == y_test).float().mean().item()
        
        train_losses.append(avg_train_loss)
        test_losses.append(test_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        
        if epoch % 10 == 0:
            print(f'Epoch {epoch}/{epochs}: Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')
        
        model.train()
    
    return train_losses, test_losses, train_accs, test_accs

# Initialize and train early fusion model
early_fusion_model = EarlyFusionModel(X_train.shape[2], X_train_graph.shape[2])

print("🚀 Training Early Fusion Model...")
ef_train_losses, ef_test_losses, ef_train_accs, ef_test_accs = train_fusion_model(
    early_fusion_model, X_train_tensor, X_train_graph_tensor, y_train_tensor,
    X_test_tensor, X_test_graph_tensor, y_test_tensor
)

## ⚡ Section 6: Late Fusion - Decision-Level Integration

Late fusion combines predictions from independently trained models, allowing each modality to develop specialized representations.

In [ ]:
# Late Fusion - Combine predictions from individual models
def late_fusion_prediction(tabular_pred, graph_pred, fusion_weights=None):
    """
    Combine predictions using weighted average
    """
    if fusion_weights is None:
        fusion_weights = [0.5, 0.5]  # Equal weights
    
    combined_pred = (fusion_weights[0] * tabular_pred + 
                    fusion_weights[1] * graph_pred)
    return combined_pred

# Get predictions from individual models
tabular_model.eval()
graph_model.eval()

with torch.no_grad():
    # Tabular predictions
    tab_train_pred = tabular_model(X_train_tensor).squeeze().numpy()
    tab_test_pred = tabular_model(X_test_tensor).squeeze().numpy()
    
    # Graph predictions  
    graph_train_pred_aligned = graph_model(X_train_graph_tensor).squeeze().numpy()
    graph_test_pred_aligned = graph_model(X_test_graph_tensor).squeeze().numpy()

# Late fusion with different weight combinations
fusion_strategies = {
    'Equal Weight': [0.5, 0.5],
    'Tabular Dominant': [0.7, 0.3],
    'Graph Dominant': [0.3, 0.7],
    'Optimal': [0.6, 0.4]  # This would typically be found through validation
}

late_fusion_results = {}

for strategy_name, weights in fusion_strategies.items():
    # Combine predictions
    lf_train_pred = late_fusion_prediction(tab_train_pred, graph_train_pred_aligned, weights)
    lf_test_pred = late_fusion_prediction(tab_test_pred, graph_test_pred_aligned, weights)
    
    # Calculate metrics
    lf_train_auc = roc_auc_score(y_train, lf_train_pred)
    lf_test_auc = roc_auc_score(y_test, lf_test_pred)
    
    lf_test_pred_binary = (lf_test_pred > 0.5).astype(int)
    lf_test_accuracy = accuracy_score(y_test, lf_test_pred_binary)
    lf_brier_score = brier_score_loss(y_test, lf_test_pred)
    
    late_fusion_results[strategy_name] = {
        'train_auc': lf_train_auc,
        'test_auc': lf_test_auc,
        'test_accuracy': lf_test_accuracy,
        'brier_score': lf_brier_score,
        'test_predictions': lf_test_pred
    }

# Display results
print("📊 Late Fusion Results:")
print("=" * 60)
for strategy, results in late_fusion_results.items():
    print(f"{strategy}:")
    print(f"  Train AUC: {results['train_auc']:.4f}")
    print(f"  Test AUC: {results['test_auc']:.4f}")
    print(f"  Test Accuracy: {results['test_accuracy']:.4f}")
    print(f"  Brier Score: {results['brier_score']:.4f}")
    print("-" * 40)

## 📈 Section 7: Comprehensive Results Analysis and Uncertainty Quantification

Let's compare all approaches and analyze uncertainty in our predictions for risk-aware decision making.

In [ ]:
# Get Early Fusion predictions
early_fusion_model.eval()
with torch.no_grad():
    ef_train_pred = early_fusion_model(X_train_tensor, X_train_graph_tensor).squeeze().numpy()
    ef_test_pred = early_fusion_model(X_test_tensor, X_test_graph_tensor).squeeze().numpy()

ef_train_auc = roc_auc_score(y_train, ef_train_pred)
ef_test_auc = roc_auc_score(y_test, ef_test_pred)
ef_test_accuracy = accuracy_score(y_test, (ef_test_pred > 0.5).astype(int))
ef_brier_score = brier_score_loss(y_test, ef_test_pred)

# Comprehensive comparison
models_comparison = {
    'Tabular Only': {
        'test_auc': test_auc,
        'test_accuracy': test_accuracy,
        'brier_score': brier_score,
        'predictions': test_predictions
    },
    'Graph Only': {
        'test_auc': graph_test_auc,
        'test_accuracy': graph_test_accuracy,
        'brier_score': graph_brier_score,
        'predictions': graph_test_pred
    },
    'Early Fusion': {
        'test_auc': ef_test_auc,
        'test_accuracy': ef_test_accuracy,
        'brier_score': ef_brier_score,
        'predictions': ef_test_pred
    },
    'Late Fusion (Optimal)': late_fusion_results['Optimal']
}

# Create comparison table
print("🏆 Model Comparison Results:")
print("=" * 80)
print(f"{'Model':<20} {'AUC':<8} {'Accuracy':<10} {'Brier Score':<12} {'Interpretation'}")
print("=" * 80)

for model_name, results in models_comparison.items():
    auc = results['test_auc']
    acc = results['test_accuracy']
    brier = results['brier_score']
    
    if auc >= 0.9:
        interpretation = "Excellent"
    elif auc >= 0.8:
        interpretation = "Good"
    elif auc >= 0.7:
        interpretation = "Fair"
    else:
        interpretation = "Poor"
    
    print(f"{model_name:<20} {auc:<8.4f} {acc:<10.4f} {brier:<12.4f} {interpretation}")

print("=" * 80)

In [ ]:
# Comprehensive Visualization
plt.figure(figsize=(20, 12))

# ROC Curves Comparison
plt.subplot(2, 4, 1)
for model_name, results in models_comparison.items():
    fpr, tpr, _ = roc_curve(y_test, results['predictions'])
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={results["test_auc"]:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')

# Performance Metrics Comparison
plt.subplot(2, 4, 2)
metrics = ['AUC', 'Accuracy', 'Brier Score (inv)']
model_names = list(models_comparison.keys())
auc_scores = [models_comparison[name]['test_auc'] for name in model_names]
acc_scores = [models_comparison[name]['test_accuracy'] for name in model_names]
brier_scores = [1 - models_comparison[name]['brier_score'] for name in model_names]  # Invert for better visualization

x = np.arange(len(model_names))
width = 0.25

plt.bar(x - width, auc_scores, width, label='AUC', alpha=0.8)
plt.bar(x, acc_scores, width, label='Accuracy', alpha=0.8)
plt.bar(x + width, brier_scores, width, label='1 - Brier Score', alpha=0.8)

plt.xlabel('Models')
plt.ylabel('Score')
plt.title('Performance Metrics Comparison')
plt.xticks(x, model_names, rotation=45)
plt.legend()

# Confusion Matrix for Best Model (find best AUC)
best_model = max(models_comparison.items(), key=lambda x: x[1]['test_auc'])
best_model_name = best_model[0]
best_predictions = best_model[1]['predictions']

plt.subplot(2, 4, 3)
cm = confusion_matrix(y_test, (best_predictions > 0.5).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix\n{best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Prediction Confidence Distribution
plt.subplot(2, 4, 4)
plt.hist(best_predictions[y_test==0], alpha=0.5, label='Trojan-Free', bins=20)
plt.hist(best_predictions[y_test==1], alpha=0.5, label='Trojan-Infected', bins=20)
plt.xlabel('Prediction Confidence')
plt.ylabel('Frequency')
plt.title(f'Prediction Confidence\n{best_model_name}')
plt.legend()

# Uncertainty Analysis
plt.subplot(2, 4, 5)
# Calculate prediction entropy (uncertainty measure)
def calculate_entropy(predictions):
    p = np.clip(predictions, 1e-8, 1-1e-8)  # Avoid log(0)
    return -(p * np.log(p) + (1-p) * np.log(1-p))

entropy_scores = calculate_entropy(best_predictions)
plt.scatter(best_predictions, entropy_scores, alpha=0.6, c=y_test, cmap='coolwarm')
plt.xlabel('Prediction Confidence')
plt.ylabel('Prediction Entropy (Uncertainty)')
plt.title('Uncertainty vs Confidence')
plt.colorbar(label='True Label')

# Feature Importance (using gradient-based method)
plt.subplot(2, 4, 6)
# Simplified feature importance for educational purposes
feature_importance = np.abs(np.corrcoef(tabular_data.iloc[:, :-1].T, tabular_data.iloc[:, -1])[:-1, -1])
plt.bar(range(len(feature_importance)), feature_importance)
plt.xlabel('Feature Index')
plt.ylabel('Correlation with Target')
plt.title('Feature Importance')

# Model Reliability Analysis
plt.subplot(2, 4, 7)
# Calibration curve
from sklearn.calibration import calibration_curve
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, best_predictions, n_bins=10
)
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label=best_model_name)
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curve')
plt.legend()

# Risk Assessment
plt.subplot(2, 4, 8)
# Define risk categories based on prediction confidence and uncertainty
high_conf_correct = ((best_predictions > 0.7) & (y_test == 1)) | ((best_predictions < 0.3) & (y_test == 0))
uncertain_predictions = (best_predictions > 0.3) & (best_predictions < 0.7)
high_conf_incorrect = ((best_predictions > 0.7) & (y_test == 0)) | ((best_predictions < 0.3) & (y_test == 1))

risk_categories = ['High Confidence\nCorrect', 'Uncertain\nPredictions', 'High Confidence\nIncorrect']
risk_counts = [np.sum(high_conf_correct), np.sum(uncertain_predictions), np.sum(high_conf_incorrect)]

colors = ['green', 'yellow', 'red']
plt.pie(risk_counts, labels=risk_categories, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Prediction Risk Assessment')

plt.tight_layout()
plt.show()

## 🎓 Section 8: Educational Summary and Key Takeaways

### What We Learned:

1. **Multimodal Learning**: Combining different data representations (tabular and graph) can improve detection accuracy
2. **Fusion Strategies**: 
   - **Early Fusion**: Combines features before learning
   - **Late Fusion**: Combines decisions from separate models
3. **Uncertainty Quantification**: Brier score and entropy help assess prediction reliability
4. **Risk Assessment**: Understanding when models are uncertain helps in critical security decisions

### Real-World Applications in Chip Design:

- **Hardware Security**: Detecting malicious modifications in integrated circuits
- **Quality Assurance**: Identifying design anomalies during verification
- **Supply Chain Security**: Verifying chip integrity from untrusted foundries
- **Design Optimization**: Using AI to improve chip security features

### Next Steps for Students:

1. Experiment with different fusion weights in late fusion
2. Try implementing actual Graph Neural Networks (GCN, GraphSAGE)
3. Explore other uncertainty quantification techniques
4. Apply similar approaches to other hardware security problems

In [ ]:
# Interactive Exercise: Let students modify fusion weights
print("🔬 Interactive Exercise: Experiment with Fusion Weights")
print("=" * 60)
print("Modify the fusion weights below and see how it affects performance:")
print()

# Interactive fusion weight exploration
def interactive_fusion_experiment(tabular_weight, graph_weight):
    """
    Allow students to experiment with different fusion weights
    """
    # Normalize weights
    total = tabular_weight + graph_weight
    w_tab = tabular_weight / total
    w_graph = graph_weight / total
    
    # Combine predictions
    combined_pred = late_fusion_prediction(tab_test_pred, graph_test_pred_aligned, [w_tab, w_graph])
    
    # Calculate metrics
    auc = roc_auc_score(y_test, combined_pred)
    accuracy = accuracy_score(y_test, (combined_pred > 0.5).astype(int))
    brier = brier_score_loss(y_test, combined_pred)
    
    print(f"Tabular Weight: {w_tab:.2f}, Graph Weight: {w_graph:.2f}")
    print(f"AUC: {auc:.4f}, Accuracy: {accuracy:.4f}, Brier Score: {brier:.4f}")
    print()
    
    return auc, accuracy, brier

# Example experiments
print("Example 1: Equal weights")
interactive_fusion_experiment(1.0, 1.0)

print("Example 2: Favor tabular data")
interactive_fusion_experiment(0.8, 0.2)

print("Example 3: Favor graph data")
interactive_fusion_experiment(0.2, 0.8)

print("💡 Try different weight combinations to find optimal performance!")
print("💡 Consider: When might you want to trust one modality over another?")

## 📚 References and Further Reading

### Original Paper:
**"NOODLE: Uncertainty-Aware Hardware Trojan Detection Using Multimodal Deep Learning"**
- Authors: Rahul Vishwakarma & Amin Rezaei
- Repository: [https://github.com/cars-lab-repo/NOODLE](https://github.com/cars-lab-repo/NOODLE)

### Related Resources:
1. **Hardware Security**: TrustHub.org - Hardware Trojan benchmarks
2. **Graph Neural Networks**: PyTorch Geometric documentation
3. **Multimodal Learning**: "Multimodal Machine Learning: A Survey" (Baltrusaitis et al.)
4. **Uncertainty Quantification**: "What Uncertainties Do We Need in Bayesian Deep Learning?" (Kendall & Gal)

### Course Context:
This notebook demonstrates how **Large Language Models and AI techniques** are revolutionizing **chip design security**:
- Traditional rule-based detection → AI-powered multimodal analysis
- Manual inspection → Automated uncertainty-aware decisions
- Single data type → Multimodal fusion approaches

---

**🎯 Assignment Ideas:**
1. Implement a different fusion strategy (attention-based fusion)
2. Add Monte Carlo Dropout for better uncertainty estimation
3. Create a real-time Trojan detection dashboard
4. Compare with traditional machine learning approaches

**Made with ❤️ for LLM4ChipDesign Course**